[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kauelopes/topicos-ml/blob/main/aulas/01-python-numpy-jax/notebooks/01_demo.ipynb)

# Aula 1 — Demonstração: Python, NumPy e JAX
*Computação Aplicada à Economia: Tópicos em Aprendizado de Máquina*

Roteiro do live coding:
1. Python essencial: tipos, coleções, `if`, `for`, `def`
2. NumPy: do laço à operação vetorizada; MQO matricial
3. JAX: `grad`, `jit`, `vmap`
4. Gráficos: do rascunho à figura de relatório (se faltar tempo, siga em casa)

## 1. Python essencial

In [ ]:
selic = 15.0          # float
n_meses = 12          # int
pais = "Brasil"       # str
acima_da_meta = True  # bool

taxa_mensal = (1 + selic / 100) ** (1 / n_meses) - 1
print(f"Selic mensal equivalente: {taxa_mensal:.4%}")
type(selic), type(n_meses)

In [ ]:
# Ponto flutuante não é exato
import math
print(0.1 + 0.2 == 0.3, math.isclose(0.1 + 0.2, 0.3))

In [ ]:
ipca = [0.16, 0.43, 0.56, 0.43]       # variação mensal (%)
metas = {"centro": 3.0, "tolerancia": 1.5}
print(ipca[0], ipca[-1], ipca[1:3])
print("Teto:", metas["centro"] + metas["tolerancia"])

In [ ]:
def acumulada(taxas_pct):
    """Inflação acumulada (%) a partir de taxas mensais (%)."""
    fator = 1.0
    for pi in taxas_pct:
        fator = fator * (1 + pi / 100)
    return 100 * (fator - 1)


def avaliar(inflacao_12m, centro=3.0, tolerancia=1.5):
    if inflacao_12m > centro + tolerancia:
        return "acima do teto"
    elif inflacao_12m < centro - tolerancia:
        return "abaixo do piso"
    return "dentro do intervalo"


acumulada(ipca), avaliar(5.2), avaliar(2.0)

## 2. NumPy: computação vetorizada

In [ ]:
import numpy as np

pi = np.array(ipca)
X = np.array([[1, 2.0], [1, 3.5], [1, 1.2]])
print(pi.shape, X.shape)
print(X[:, 1])
print(X[X[:, 1] > 1.5])

### Laço × vetorizado

In [ ]:
rng = np.random.default_rng(0)
taxas = rng.normal(0.4, 0.3, size=100_000)   # com 1 milhão de meses, o produto estoura o float

%timeit acumulada(taxas)
%timeit 100 * (np.prod(1 + taxas / 100) - 1)

### Broadcasting: padronizar colunas

In [ ]:
X_pad = (X - X.mean(axis=0)) / np.where(X.std(axis=0) > 0, X.std(axis=0), 1)
X.mean(axis=0).shape, X_pad

### MQO em uma linha
Equações normais: $(X^\top X)\hat\beta = X^\top y$. **Resolva o sistema**, não inverta a matriz.

In [ ]:
rng = np.random.default_rng(42)
n = 500
x = rng.normal(size=n)
X = np.column_stack([np.ones(n), x])
y = 1.0 + 2.0 * x + rng.normal(size=n)

beta = np.linalg.solve(X.T @ X, X.T @ y)
beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
beta, beta_lstsq

## 3. JAX

In [ ]:
import jax
import jax.numpy as jnp

x = jnp.arange(5.0)
# x[0] = 10.0          # erro: arrays JAX são imutáveis
x = x.at[0].set(10.0)
x

### `grad`: utilidade marginal CRRA, $u(c)=c^{1-\gamma}/(1-\gamma)$

In [ ]:
gamma = 2.0
def u(c):
    return c ** (1 - gamma) / (1 - gamma)

du = jax.grad(u)
print(du(1.5), 1.5 ** -gamma)

c = jnp.linspace(0.5, 3.0, 6)
jax.vmap(du)(c), c ** -gamma

### Gradiente da perda quadrática
$\mathcal L(\beta) = \frac1n\lVert y - X\beta\rVert^2$, $\quad \nabla_\beta \mathcal L = -\frac2n X^\top(y - X\beta)$

In [ ]:
def perda(beta, X, y):
    r = y - X @ beta
    return jnp.mean(r ** 2)

grad_perda = jax.grad(perda)
b0 = jnp.zeros(2)
print(grad_perda(b0, X, y))
print(-2 / n * X.T @ (y - X @ b0))
print("No MQO:", grad_perda(beta, X, y))   # ~ 0

### `jit`: compilar

In [ ]:
grad_rapido = jax.jit(grad_perda)
grad_rapido(b0, X, y).block_until_ready()   # 1ª chamada: compila

%timeit grad_perda(b0, X, y).block_until_ready()
%timeit grad_rapido(b0, X, y).block_until_ready()

## 4. Gráficos: do rascunho à figura de relatório
Vamos desenhar o IPCA acumulado em 12 meses e melhorar o gráfico **passo a passo**. Cada passo usa um recurso do Python que já vimos (dicionários, laços, f-strings, máscaras booleanas, funções). Os dados vêm da API do IBGE (SIDRA, tabela 1737).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

URL_SIDRA = "https://apisidra.ibge.gov.br/values/t/1737/n1/all/v/63/p/all"   # IBGE: IPCA mensal (%)

try:
    tabela = pd.read_json(URL_SIDRA).iloc[1:]           # a 1ª linha da resposta é o cabeçalho
    tabela = tabela[tabela["D3C"] >= "201001"]          # de 2010 em diante
    ipca = pd.Series(tabela["V"].astype(float).to_numpy(),
                     index=pd.to_datetime(tabela["D3C"], format="%Y%m"), name="ipca")
    print(f"IPCA do IBGE: {ipca.index[0]:%m/%Y} a {ipca.index[-1]:%m/%Y}")
except Exception as erro:  # sem internet: série simulada para a aula não travar
    print("Sem acesso ao IBGE, usando dados simulados:", erro)
    rng = np.random.default_rng(1)
    datas = pd.date_range("2010-01-01", periods=200, freq="MS")
    ipca = pd.Series(np.round(0.4 + 0.3 * rng.standard_normal(200), 2), index=datas, name="ipca")

# Acumulado em 12 meses encadeando métodos: cada .método() devolve um objeto novo
ipca12 = ((1 + ipca / 100).rolling(12).apply(np.prod, raw=True) - 1).mul(100).dropna()
ipca12.tail()

### Passo 1: o rascunho
Funciona, mas não tem título, unidade nem contexto. Quem olha não sabe o que é.

In [ ]:
plt.plot(ipca12)
plt.show()

### Passo 2: `fig, ax` e rótulos
A interface orientada a objetos dá controle sobre cada parte. Recurso do Python: **desempacotar um dicionário** com `**`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))   # fig = a folha; ax = o gráfico dentro dela
ax.plot(ipca12.index, ipca12.values)

rotulos = dict(title="IPCA acumulado em 12 meses", ylabel="% em 12 meses")
ax.set(**rotulos)                            # ** desempacota o dicionário em argumentos nomeados
plt.show()

### Passo 3: estilo
O matplotlib traz estilos prontos. O `with` aplica um estilo **só dentro do bloco**.

In [ ]:
print(plt.style.available[:6], "...")      # estilos prontos (fatiamento de lista)

for estilo in ["ggplot", "seaborn-v0_8-whitegrid", "fivethirtyeight"]:
    with plt.style.context(estilo):          # 'with': o estilo vale só dentro do bloco
        fig, ax = plt.subplots(figsize=(6, 2.4))
        ax.plot(ipca12)
        ax.set_title(f"estilo: {estilo}")
        plt.show()

Melhor ainda: um estilo **nosso**, definido uma vez com dicionários. Todos os gráficos seguintes o herdam.

In [ ]:
# Nosso próprio estilo: dois dicionários
CORES = {"azul": "#2A6F97", "laranja": "#EB811B", "verde": "#14B03D",
         "vermelho": "#B22234", "escuro": "#23373B", "cinza": "#8C8C8C"}

plt.rcParams.update({
    "figure.figsize": (9, 3.8),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.prop_cycle": plt.cycler(color=list(CORES.values())),
})

### Passo 4: contexto econômico
Uma série de inflação sem a meta não diz se 6% é muito ou pouco. Recursos: **dict comprehension**, **list comprehension** e **máscara booleana** para pintar os meses acima do teto.

In [ ]:
# Metas para a inflação (CMN): ano -> (centro, tolerância), em %
metas = {ano: (4.5, 2.0) for ano in range(2010, 2017)}         # dict comprehension
metas.update({2017: (4.5, 1.5), 2018: (4.5, 1.5), 2019: (4.25, 1.5), 2020: (4.0, 1.5),
              2021: (3.75, 1.5), 2022: (3.5, 1.5), 2023: (3.25, 1.5)})
metas.update({ano: (3.0, 1.5) for ano in range(2024, 2027)})  # meta contínua a partir de 2025

# Para cada mês, buscamos a meta do ano (list comprehension + índice de tupla)
centro = pd.Series([metas[d.year][0] for d in ipca12.index], index=ipca12.index)
tol = pd.Series([metas[d.year][1] for d in ipca12.index], index=ipca12.index)

fig, ax = plt.subplots()
ax.fill_between(ipca12.index, centro - tol, centro + tol, step="post",
                color=CORES["verde"], alpha=0.15)
ax.plot(centro, color=CORES["verde"], lw=1, ls="--", drawstyle="steps-post")
ax.plot(ipca12, color=CORES["azul"], lw=2.2)

acima = ipca12 > centro + tol                # máscara booleana, como no NumPy
ax.fill_between(ipca12.index, centro + tol, ipca12, where=acima, interpolate=True,
                color=CORES["vermelho"], alpha=0.3)
ax.set(title="IPCA em 12 meses e a meta de inflação", ylabel="%")
plt.show()

print(f"Meses acima do teto: {acima.sum()} de {len(acima)} ({acima.mean():.0%})")

### Passo 5: eixos legíveis, anotação e uma função reutilizável
Formatadores de eixo, `annotate` com seta, f-string que formata datas (`{data:%m/%Y}`) e rótulos diretos no lugar da legenda. Tudo dentro de uma **função**, para reaproveitar.

In [ ]:
from matplotlib.ticker import PercentFormatter
import matplotlib.dates as mdates


def grafico_ipca(serie, centro, tol, ax=None, cor=CORES["azul"]):
    """IPCA em 12 meses com a banda da meta. Devolve o eixo para ajustes."""
    if ax is None:                           # argumento opcional: cria o eixo se não vier um
        fig, ax = plt.subplots()
    acima = serie > centro + tol
    ax.fill_between(serie.index, centro - tol, centro + tol, step="post",
                    color=CORES["verde"], alpha=0.15)
    ax.fill_between(serie.index, centro + tol, serie, where=acima, interpolate=True,
                    color=CORES["vermelho"], alpha=0.25)
    ax.plot(serie, color=cor, lw=2.2)

    # Eixos: % no y; um rótulo a cada 2 anos no x
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # Anotar o pico: idxmax devolve a DATA do maior valor; f-string formata a data
    data_pico = serie.idxmax()
    ax.annotate(f"pico: {serie.max():.1f}% em {data_pico:%m/%Y}",
                xy=(data_pico, serie.max()), xytext=(25, -5), textcoords="offset points",
                va="center", arrowprops=dict(arrowstyle="->", color=CORES["escuro"]))

    # Rótulos diretos no lugar da legenda
    ax.text(serie.index[-1], serie.iloc[-1], f"  {serie.iloc[-1]:.1f}%",
            color=cor, fontweight="bold", va="center")
    ax.text(serie.index[0], (centro - tol).iloc[0], " intervalo da meta",
            color=CORES["verde"], va="top", fontsize=9)
    return ax


ax = grafico_ipca(ipca12, centro, tol)
ax.set(title="IPCA em 12 meses: acima do teto, em vermelho")
plt.show()

### Passo 6: dois painéis e exportação
Mensal em cima, acumulado embaixo, com o mesmo eixo de datas. Recursos: `np.where` para escolher cores e `savefig` para exportar.

In [ ]:
mensal = ipca[ipca12.index[0]:]                       # fatiamento por data
cores = np.where(mensal < 0, CORES["verde"], CORES["azul"])   # deflação em verde

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True,
                               gridspec_kw={"height_ratios": [1, 2]})
ax1.bar(mensal.index, mensal, width=25, color=cores)
ax1.set(title="Variação mensal", ylabel="% no mês")
grafico_ipca(ipca12, centro, tol, ax=ax2)            # reaproveita a função
ax2.set(title="Acumulado em 12 meses")
fig.suptitle("IPCA — Brasil", fontweight="bold", x=0.02, ha="left")
fig.tight_layout()
fig.savefig("ipca.png", dpi=200, bbox_inches="tight")  # pronto para o relatório
plt.show()

### Resumo: do rascunho à figura de relatório

| Recurso | Para quê |
|---|---|
| `fig, ax = plt.subplots()` | controlar cada parte do gráfico |
| `ax.set(**rotulos)` | dicionário desempacotado em argumentos nomeados |
| `with plt.style.context(...)` | testar um estilo só dentro do bloco |
| `plt.rcParams.update({...})` | definir o padrão de todos os gráficos de uma vez |
| `{ano: ... for ano in range(...)}` | montar dicionários em uma linha |
| `fill_between(..., where=mascara)` | destacar períodos com uma máscara booleana |
| `PercentFormatter`, `mdates` | eixos legíveis: %, anos |
| `f"{data:%m/%Y}"` | f-string que formata datas |
| `ax.annotate`, `ax.text` | contar a história no próprio gráfico |
| `def grafico(..., ax=None)` | reaproveitar o gráfico em vários painéis |
| `fig.savefig(..., dpi=200)` | exportar com qualidade |